# Release16 Failed-Fold State Feature Mining

This notebook follows `release16_remaining_failedfold_diagnostic_v1`: the overlay improved `wf_2020` but still missed the relative-return gate, while `wf_2023_2024` regressed. The next question is whether the remaining low-position up-day failures are separable by stable state features before another training branch is started.

Experiment governance:
- Baselines: canonical `retained_soxl_harness_model_frontier`; direct comparison `release16_remaining_failedfold_diagnostic_v1`; reference baseline `soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02`.
- Role: `diagnose`.
- Target window: failed walk-forward test folds `wf_2020` and `wf_2023_2024`; guard fold `wf_2022` remains pass evidence and is not used to tune the failed-state head.
- Changed layer: `feature`, `representation`, `target_design`, `model_selection`.
- External reference: SOXLBacktest buy/sell baseline plus existing qlib state-objective / representation diagnostics.
- Hypothesis: missed low-position up days are not a generic participation-floor problem; they need a state head that separates failed-fold rally states with stable feature groups before reward/target training.
- Acceptance criteria: identify a feature group with fold-heldout AUC >= 0.60 and accuracy >= 0.55 across failed folds; keep actor retention disabled; write sidecars and a summary usable by the retention frontier.
- Failure mode: no stable feature group separates the failed-fold low-position up days, so next work must expand features or revisit labels before training.
- Comparison-table row key: `release16_failedfold_state_feature_mining_v1`.

Historical context:
- Closest success: `rebound_state_factor_target_repair_screen_v1` created non-leaking fold-local state targets and passed BC readiness gates.
- Closest failure: `release16_failedfold_participation_target_overlay_v1` increased failed-fold training participation targets but did not produce a passing checkpoint screen.
- This is not a duplicate because it scores failed test-state feature separability before proposing another target overlay or PPO run.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'soxl_quant').exists() and (candidate / 'AGENTS.md').exists():
            return candidate
    raise RuntimeError(f'could not find repo root from {start}')

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.representation_diagnostics import score_binary_representation_feature_sets
from src.soxl_quant.data import build_feature_frame, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_state_feature_mining_v1'
CARD.mkdir(parents=True, exist_ok=True)

SOURCE_DIAGNOSTIC = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_remaining_failedfold_diagnostic_v1'
SOURCE_CHECKPOINT = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_overlay_checkpoint_screen_v1'
SOURCE_TARGET = ROOT / 'runtime' / 'model_selection' / 'card' / 'release16_failedfold_participation_target_overlay_v1'

EXPERIMENT = 'release16_failedfold_state_feature_mining_v1'
FAILED_FOLDS = ['wf_2020', 'wf_2023_2024']
GUARD_FOLDS = ['wf_2022']
ALL_FOLDS = ['wf_2020', 'wf_2022', 'wf_2023_2024']
PROFILE_NAME = 'balanced_rule_residual_v22'
FEATURE_PACKS = ('rulebase_signals', 'academic_signals', 'transition_signals')

SIDECARS = {
    'test_state_frame': CARD / 'missed_up_day_state_frame.csv',
    'test_state_feature_set_scores': CARD / 'test_state_feature_set_scores.csv',
    'test_state_fold_metrics': CARD / 'test_state_fold_metrics.csv',
    'test_state_feature_effects': CARD / 'test_state_feature_effects.csv',
    'train_overlay_frame': CARD / 'train_overlay_state_frame.csv',
    'train_overlay_feature_set_scores': CARD / 'train_overlay_feature_set_scores.csv',
    'train_overlay_fold_metrics': CARD / 'train_overlay_fold_metrics.csv',
    'train_overlay_feature_effects': CARD / 'train_overlay_feature_effects.csv',
    'state_transfer_feature_alignment': CARD / 'state_transfer_feature_alignment.csv',
    'fold_signal_coverage': CARD / 'fold_signal_coverage.csv',
    'candidate_feature_groups': CARD / 'candidate_feature_groups.json',
    'selection': CARD / 'selection.json',
    'summary': CARD / 'summary.json',
    'conclusion': CARD / 'conclusion.md',
}

def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, payload: dict[str, Any]) -> None:
    def clean(value: Any) -> Any:
        if isinstance(value, dict):
            return {str(k): clean(v) for k, v in value.items()}
        if isinstance(value, list):
            return [clean(v) for v in value]
        if isinstance(value, tuple):
            return [clean(v) for v in value]
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, (np.floating,)):
            return None if not np.isfinite(value) else float(value)
        if isinstance(value, float):
            return None if not np.isfinite(value) else value
        if isinstance(value, (pd.Timestamp,)):
            return value.date().isoformat()
        return value
    path.write_text(json.dumps(clean(payload), indent=2, sort_keys=True), encoding='utf-8')

def pct_return(values: pd.Series) -> float:
    first = float(values.iloc[0])
    last = float(values.iloc[-1])
    return (last / first - 1.0) * 100.0 if first else 0.0

def max_drawdown(values: pd.Series) -> float:
    arr = values.to_numpy(dtype=float)
    peaks = np.maximum.accumulate(arr)
    dd = arr / np.maximum(peaks, 1e-12) - 1.0
    return abs(float(np.nanmin(dd)) * 100.0)

def as_date_series(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series).dt.normalize()

source_summary = load_json(SOURCE_DIAGNOSTIC / 'summary.json')
source_selection = load_json(SOURCE_DIAGNOSTIC / 'selection.json')
fold_gap = pd.read_csv(SOURCE_DIAGNOSTIC / 'fold_gate_gap_summary.csv')
missed = pd.read_csv(SOURCE_DIAGNOSTIC / 'missed_up_days_after_overlay.csv')
combined_targets = pd.read_csv(SOURCE_TARGET / 'combined_target_contract.csv')

print(json.dumps({
    'source_decision': source_selection['decision'],
    'source_next': source_selection['next_mechanism'],
    'failed_folds': source_selection['failed_folds'],
    'target_rows': len(combined_targets),
    'missed_up_days': len(missed),
}, indent=2))

{
  "source_decision": "release16_overlay_failure_is_state_transfer_and_representation_gap_no_promotion",
  "source_next": "mine_failedfold_test_state_features_then_rebuild_representation_or_reward_objective_before_training",
  "failed_folds": [
    "wf_2020",
    "wf_2023_2024"
  ],
  "target_rows": 3731,
  "missed_up_days": 113
}


In [2]:
# Rebuild the current qlib feature frame from repo-owned data.
init_qlib(QLIB_DATA_PATH)
features, price = build_feature_frame(
    '2010-01-01',
    '2024-12-31',
    feature_packs=FEATURE_PACKS,
    profile_name=PROFILE_NAME,
)
feature_frame = features.droplevel('instrument').reset_index().copy()
feature_frame['datetime'] = pd.to_datetime(feature_frame['datetime']).dt.normalize()
feature_frame = feature_frame.sort_values('datetime').reset_index(drop=True)
print({
    'rows': len(feature_frame),
    'cols': len(feature_frame.columns),
    'start': feature_frame['datetime'].min().date().isoformat(),
    'end': feature_frame['datetime'].max().date().isoformat(),
    'profile_name': PROFILE_NAME,
    'feature_packs': FEATURE_PACKS,
})
feature_frame.head(3)

[15793:MainThread](2026-06-14 23:50:29,114) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[15793:MainThread](2026-06-14 23:50:29,542) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[15793:MainThread](2026-06-14 23:50:29,542) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


{'rows': 3475, 'cols': 92, 'start': '2011-03-11', 'end': '2024-12-31', 'profile_name': 'balanced_rule_residual_v22', 'feature_packs': ('rulebase_signals', 'academic_signals', 'transition_signals')}


,datetime,return,anchor_gap_180,anchor_slope_180,trend_gap_21,trend_gap_63,trend_slope_21,trend_slope_63,rsi_14,mom_21,...,TRANS_SPRING_COMPRESSION,TRANS_PANIC_EXHAUSTION,TRANS_GAP_RECOVERY,TRANS_VOL_TERM_ZSCORE,TRANS_VOV_TREND,TRANS_TE_ZSCORE,INTRADAY_REJECTION_EVENT,SIGNED_TE_REVERSION_5,SIGNED_TE_REVERSION_10,SIGNED_TE_REVERSION_21
0,2011-03-11,0.026661,0.260999,0.089819,-0.125094,0.097657,-0.024845,0.134495,0.309724,-0.194437,...,0.0,1.0,1.0,0.625364,0.023407,0.189161,0.000000,0.0,0.0,0.0
1,2011-03-14,-0.004081,0.252652,0.087180,-0.141470,0.085597,-0.036711,0.126186,0.370506,-0.207096,...,0.0,0.0,0.0,0.628394,0.031876,-0.041545,0.000000,0.0,0.0,0.0
2,2011-03-15,-0.036320,0.204454,0.084765,-0.163109,0.071038,-0.051835,0.116713,0.384049,-0.266289,...,0.0,0.0,0.0,0.530114,0.034527,0.070963,0.332358,0.0,0.0,0.0


In [3]:
CANDIDATE_FEATURE_GROUPS = {
    'momentum_drawdown': [
        'mom_5', 'mom_21', 'mom_63', 'drawdown_21', 'drawdown_63',
        'trend_gap_21', 'trend_gap_63', 'anchor_gap_180', 'anchor_slope_180',
    ],
    'volatility_stress': [
        'realized_vol_10', 'realized_vol_21', 'realized_vol_63', 'vol_of_vol_21',
        'downside_ratio_21', 'downside_semivol_21', 'vol_coef_21',
        'realized_kurtosis_21', 'vol_stress_score', 'regime_risk',
    ],
    'rulebase_reversal': [
        'RB_PANIC_SCORE_PROXY', 'RB_DAILY_DROP_SEVERITY', 'RB_CUMULATIVE_DROP_SEVERITY',
        'RB_RSI6_OVERSOLD', 'RB_VOL_REGIME', 'RB_BUY_COMPOSITE', 'RB_SELL_COMPOSITE',
        'RB_PROFIT_TAKE_PROXY', 'RB_VOLUME_STRESS', 'RB_TREND_REVERSAL',
    ],
    'transition_recovery': [
        'TRANS_VIX_DIVERGENCE', 'TRANS_SPRING_COMPRESSION', 'TRANS_PANIC_EXHAUSTION',
        'TRANS_GAP_RECOVERY', 'TRANS_VOL_TERM_ZSCORE', 'TRANS_VOV_TREND',
        'TRANS_TE_ZSCORE', 'INTRADAY_REJECTION_EVENT', 'SIGNED_TE_REVERSION_5',
    ],
    'soxx_relative_decay': [
        'tracking_error_5', 'tracking_error_21', 'tracking_error_vol_21',
        'decay_63', 'relative_strength_21', 'ACAD_BETA_DEVIATION',
        'ACAD_VOLATILITY_DRAG', 'ACAD_AUTOCORR_5',
    ],
}
CANDIDATE_FEATURE_GROUPS['compact_state_head'] = [
    'mom_5', 'mom_21', 'drawdown_63', 'vol_of_vol_21', 'vol_coef_21',
    'relative_strength_21', 'RB_BUY_COMPOSITE', 'RB_TREND_REVERSAL',
    'TRANS_PANIC_EXHAUSTION', 'TRANS_GAP_RECOVERY', 'SIGNED_TE_REVERSION_5',
]
CANDIDATE_FEATURE_GROUPS = {
    name: [feature for feature in features if feature in feature_frame.columns]
    for name, features in CANDIDATE_FEATURE_GROUPS.items()
}
write_json(SIDECARS['candidate_feature_groups'], CANDIDATE_FEATURE_GROUPS)
print(json.dumps(CANDIDATE_FEATURE_GROUPS, indent=2))

{
  "momentum_drawdown": [
    "mom_5",
    "mom_21",
    "mom_63",
    "drawdown_21",
    "drawdown_63",
    "trend_gap_21",
    "trend_gap_63",
    "anchor_gap_180",
    "anchor_slope_180"
  ],
  "volatility_stress": [
    "realized_vol_10",
    "realized_vol_21",
    "realized_vol_63",
    "vol_of_vol_21",
    "downside_ratio_21",
    "downside_semivol_21",
    "vol_coef_21",
    "realized_kurtosis_21",
    "vol_stress_score",
    "regime_risk"
  ],
  "rulebase_reversal": [
    "RB_PANIC_SCORE_PROXY",
    "RB_DAILY_DROP_SEVERITY",
    "RB_CUMULATIVE_DROP_SEVERITY",
    "RB_RSI6_OVERSOLD",
    "RB_VOL_REGIME",
    "RB_BUY_COMPOSITE",
    "RB_SELL_COMPOSITE",
    "RB_PROFIT_TAKE_PROXY",
    "RB_VOLUME_STRESS",
    "RB_TREND_REVERSAL"
  ],
  "transition_recovery": [
    "TRANS_VIX_DIVERGENCE",
    "TRANS_SPRING_COMPRESSION",
    "TRANS_PANIC_EXHAUSTION",
    "TRANS_GAP_RECOVERY",
    "TRANS_VOL_TERM_ZSCORE",
    "TRANS_VOV_TREND",
    "TRANS_TE_ZSCORE",
    "INTRADAY_REJECTION_EVENT",


In [4]:
def selected_daily_for_fold(fold: str) -> pd.DataFrame:
    path = SOURCE_CHECKPOINT / 'official_daily_paths' / f'{fold}__bc_initialization_official_daily_path.csv'
    daily = pd.read_csv(path)
    daily['datetime'] = pd.to_datetime(daily['datetime']).dt.normalize()
    daily['fold'] = fold
    daily = daily.sort_values('datetime').reset_index(drop=True)
    daily['selected_prev_position'] = daily['position'].shift(1).fillna(daily['position'])
    daily['soxl_daily_return'] = daily['price'].pct_change().fillna(0.0)
    daily['selected_low_position_threshold'] = 0.20
    daily['up_day_threshold'] = 0.04
    return daily

missed['datetime'] = pd.to_datetime(missed['datetime']).dt.normalize()
missed_keys = set(zip(missed['fold'].astype(str), missed['datetime']))

state_rows = []
for fold in ALL_FOLDS:
    daily = selected_daily_for_fold(fold)
    merged = daily.merge(feature_frame, on='datetime', how='left', validate='one_to_one')
    merged['has_feature_frame'] = merged['mom_21'].notna() if 'mom_21' in merged.columns else False
    merged['is_failed_fold'] = merged['fold'].isin(FAILED_FOLDS)
    merged['is_missed_up_day'] = [
        (fold_value, dt_value) in missed_keys
        for fold_value, dt_value in zip(merged['fold'].astype(str), merged['datetime'])
    ]
    merged['is_low_position_up_day_rule'] = (
        (merged['selected_prev_position'] <= merged['selected_low_position_threshold'])
        & (merged['soxl_daily_return'] >= merged['up_day_threshold'])
    )
    merged['missed_label_matches_rule'] = merged['is_missed_up_day'] == merged['is_low_position_up_day_rule']
    state_rows.append(merged)

test_state = pd.concat(state_rows, ignore_index=True)
test_state.to_csv(SIDECARS['test_state_frame'], index=False)
coverage = test_state.groupby('fold').agg(
    rows=('datetime', 'count'),
    feature_rows=('has_feature_frame', 'sum'),
    missed_up_days=('is_missed_up_day', 'sum'),
    low_position_up_day_rule_rows=('is_low_position_up_day_rule', 'sum'),
    label_rule_match_rate=('missed_label_matches_rule', 'mean'),
    mean_prev_position=('selected_prev_position', 'mean'),
    mean_position=('position', 'mean'),
    return_pct=('strategy_nav', pct_return),
    buyhold_return_pct=('buyhold_nav', pct_return),
    max_dd_pct=('strategy_nav', max_drawdown),
).reset_index()
coverage.to_csv(SIDECARS['fold_signal_coverage'], index=False)
coverage

,fold,rows,feature_rows,missed_up_days,low_position_up_day_rule_rows,label_rule_match_rate,mean_prev_position,mean_position,return_pct,buyhold_return_pct,max_dd_pct
0,wf_2020,252,252,47,47,1.000,0.230806,0.230203,69.514153,58.093571,16.652151
1,wf_2022,250,250,0,67,0.732,0.081004,0.081384,-14.873092,-86.588072,15.617043
2,wf_2023_2024,251,251,66,66,1.000,0.082842,0.083241,3.049745,-0.356635,5.546722


In [5]:
failed_test = test_state[test_state['fold'].isin(FAILED_FOLDS) & test_state['has_feature_frame']].copy()
result_test = score_binary_representation_feature_sets(
    failed_test,
    label_col='is_missed_up_day',
    fold_col='fold',
    feature_sets=CANDIDATE_FEATURE_GROUPS,
    min_positive_train=5,
    min_positive_eval=5,
    min_holdout_auc=0.60,
    min_holdout_accuracy=0.55,
)
result_test.feature_set_scores.to_csv(SIDECARS['test_state_feature_set_scores'], index=False)
result_test.fold_metrics.to_csv(SIDECARS['test_state_fold_metrics'], index=False)
result_test.feature_effects.to_csv(SIDECARS['test_state_feature_effects'], index=False)
result_test.feature_set_scores

,feature_set,features,feature_count,folds,covered_folds,coverage_gate,holdout_auc_min,holdout_auc_mean,holdout_accuracy_min,holdout_accuracy_mean,representation_gate
0,rulebase_reversal,"RB_PANIC_SCORE_PROXY,RB_DAILY_DROP_SEVERITY,RB...",10,2,2,True,0.863622,0.881115,0.630952,0.733803,True
1,transition_recovery,"TRANS_VIX_DIVERGENCE,TRANS_SPRING_COMPRESSION,...",9,2,2,True,0.603425,0.625177,0.678571,0.697851,True
2,compact_state_head,"mom_5,mom_21,drawdown_63,vol_of_vol_21,vol_coe...",11,2,2,True,0.690711,0.710916,0.539683,0.590558,False
3,momentum_drawdown,"mom_5,mom_21,mom_63,drawdown_21,drawdown_63,tr...",9,2,2,True,0.625117,0.648267,0.460317,0.522987,False
4,volatility_stress,"realized_vol_10,realized_vol_21,realized_vol_6...",10,2,2,True,0.542342,0.559028,0.454183,0.520742,False
5,soxx_relative_decay,"tracking_error_5,tracking_error_21,tracking_er...",8,2,2,True,0.376495,0.418242,0.541833,0.590361,False


In [6]:
# Build the train-target overlay frame to compare the feature profile that training actually saw.
targets = combined_targets.copy()
targets['datetime'] = pd.to_datetime(targets['datetime']).dt.normalize()
targets['fold_family'] = targets['fold'].astype(str).str.replace('_train', '', regex=False)
for col in ['failedfold_participation_target_position', 'source_risk_target_position', 'risk_target_position', 'teacher_position']:
    if col in targets.columns:
        targets[col] = pd.to_numeric(targets[col], errors='coerce')

target_position_col = 'failedfold_participation_target_position'
source_position_col = 'source_risk_target_position'
targets['state_target_position'] = targets[target_position_col].fillna(targets.get('risk_target_position'))
targets['default_target_position'] = targets[source_position_col].fillna(targets.get('teacher_position'))
targets['state_target_lift'] = targets['state_target_position'] - targets['default_target_position']
targets['is_overlay_lift'] = (targets['state_target_lift'] > 0.01) & targets.get('failedfold_participation_overlay_state', False).fillna(False).astype(bool)
targets['is_high_participation_target'] = targets['state_target_position'] >= 0.50

feature_cols = sorted({feature for group in CANDIDATE_FEATURE_GROUPS.values() for feature in group})
# Some target sidecars already contain feature column names, but older rows can be blank.
# Use the current repo-owned qlib feature frame consistently for representation scoring.
targets = targets.drop(columns=[feature for feature in feature_cols if feature in targets.columns], errors='ignore')
targets = targets.merge(feature_frame[['datetime', *feature_cols]], on='datetime', how='left')

train_overlay = targets[targets['fold_family'].isin(FAILED_FOLDS) | targets['fold_family'].isin(GUARD_FOLDS)].copy()
train_overlay['has_feature_frame'] = train_overlay[feature_cols].notna().all(axis=1)
train_overlay.to_csv(SIDECARS['train_overlay_frame'], index=False)
train_overlay.groupby('fold_family').agg(
    rows=('datetime', 'count'),
    feature_rows=('has_feature_frame', 'sum'),
    overlay_lift_rows=('is_overlay_lift', 'sum'),
    high_target_rows=('is_high_participation_target', 'sum'),
    mean_target=('state_target_position', 'mean'),
    mean_default=('default_target_position', 'mean'),
    mean_lift=('state_target_lift', 'mean'),
).reset_index()

,fold_family,rows,feature_rows,overlay_lift_rows,high_target_rows,mean_target,mean_default,mean_lift
0,wf_2020,2470,2217,698,749,0.357942,0.312066,0.045877
1,wf_2022,505,505,0,34,0.198093,0.198093,0.000000
2,wf_2023_2024,756,756,193,227,0.236502,0.154589,0.081913


In [7]:
failed_train_overlay = train_overlay[train_overlay['fold_family'].isin(FAILED_FOLDS) & train_overlay['has_feature_frame']].copy()
result_train = score_binary_representation_feature_sets(
    failed_train_overlay,
    label_col='is_overlay_lift',
    fold_col='fold_family',
    feature_sets=CANDIDATE_FEATURE_GROUPS,
    min_positive_train=20,
    min_positive_eval=20,
    min_holdout_auc=0.60,
    min_holdout_accuracy=0.55,
)
result_train.feature_set_scores.to_csv(SIDECARS['train_overlay_feature_set_scores'], index=False)
result_train.fold_metrics.to_csv(SIDECARS['train_overlay_fold_metrics'], index=False)
result_train.feature_effects.to_csv(SIDECARS['train_overlay_feature_effects'], index=False)
result_train.feature_set_scores

,feature_set,features,feature_count,folds,covered_folds,coverage_gate,holdout_auc_min,holdout_auc_mean,holdout_accuracy_min,holdout_accuracy_mean,representation_gate
0,momentum_drawdown,"mom_5,mom_21,mom_63,drawdown_21,drawdown_63,tr...",9,2,2,True,0.676943,0.806597,0.604871,0.695293,True
1,rulebase_reversal,"RB_PANIC_SCORE_PROXY,RB_DAILY_DROP_SEVERITY,RB...",10,2,2,True,0.666731,0.788731,0.576004,0.695409,True
2,compact_state_head,"mom_5,mom_21,drawdown_63,vol_of_vol_21,vol_coe...",11,2,2,True,0.661843,0.791035,0.579612,0.676711,True
3,soxx_relative_decay,"tracking_error_5,tracking_error_21,tracking_er...",8,2,2,True,0.612354,0.720685,0.565178,0.671478,True
4,volatility_stress,"realized_vol_10,realized_vol_21,realized_vol_6...",10,2,2,True,0.660110,0.729392,0.437077,0.591554,False
5,transition_recovery,"TRANS_VIX_DIVERGENCE,TRANS_SPRING_COMPRESSION,...",9,2,2,True,0.608788,0.650178,0.495264,0.555172,False


In [8]:
test_effects = result_test.feature_effects.rename(columns={
    'positive_mean': 'test_missed_mean',
    'negative_mean': 'test_other_mean',
    'effect_size': 'test_effect_size',
    'positive_count': 'test_positive_count',
    'negative_count': 'test_negative_count',
})
train_effects = result_train.feature_effects.rename(columns={
    'positive_mean': 'train_overlay_mean',
    'negative_mean': 'train_other_mean',
    'effect_size': 'train_effect_size',
    'positive_count': 'train_positive_count',
    'negative_count': 'train_negative_count',
})
alignment = test_effects.merge(
    train_effects,
    on=['feature_set', 'feature'],
    how='outer',
)
alignment['effect_sign_aligned'] = np.sign(alignment['test_effect_size']) == np.sign(alignment['train_effect_size'])
alignment['effect_abs_gap'] = (alignment['test_effect_size'] - alignment['train_effect_size']).abs()
alignment['test_abs_effect'] = alignment['test_effect_size'].abs()
alignment['train_abs_effect'] = alignment['train_effect_size'].abs()
alignment['joint_abs_effect'] = alignment[['test_abs_effect', 'train_abs_effect']].min(axis=1)
alignment = alignment.sort_values(['effect_sign_aligned', 'joint_abs_effect', 'test_abs_effect'], ascending=[False, False, False]).reset_index(drop=True)
alignment.to_csv(SIDECARS['state_transfer_feature_alignment'], index=False)
alignment.head(30)

,feature_set,feature,test_missed_mean,test_other_mean,test_effect_size,test_positive_count,test_negative_count,train_overlay_mean,train_other_mean,train_effect_size,train_positive_count,train_negative_count,effect_sign_aligned,effect_abs_gap,test_abs_effect,train_abs_effect,joint_abs_effect
0,rulebase_reversal,RB_SELL_COMPOSITE,0.340337,0.230796,0.489719,113,390,0.406805,0.247990,0.711590,832,2141,True,0.221871,0.489719,0.711590,0.489719
1,compact_state_head,mom_5,0.071091,0.002756,0.467311,113,390,0.043681,-0.001647,0.407630,832,2141,True,0.059682,0.467311,0.407630,0.407630
2,momentum_drawdown,mom_5,0.071091,0.002756,0.467311,113,390,0.043681,-0.001647,0.407630,832,2141,True,0.059682,0.467311,0.407630,0.407630
3,rulebase_reversal,RB_RSI6_OVERSOLD,0.003792,0.058347,-0.400469,113,390,0.021958,0.081791,-0.359886,832,2141,True,0.040583,0.400469,0.359886,0.359886
4,compact_state_head,RB_TREND_REVERSAL,0.520335,0.192729,0.357359,113,390,0.455332,0.007169,0.678399,832,2141,True,0.321041,0.357359,0.678399,0.357359
5,rulebase_reversal,RB_TREND_REVERSAL,0.520335,0.192729,0.357359,113,390,0.455332,0.007169,0.678399,832,2141,True,0.321041,0.357359,0.678399,0.357359
6,rulebase_reversal,RB_CUMULATIVE_DROP_SEVERITY,-0.090494,-0.119925,0.328998,113,390,-0.055529,-0.129126,0.713118,832,2141,True,0.384120,0.328998,0.713118,0.328998
7,momentum_drawdown,anchor_gap_180,0.300780,0.164515,0.325367,113,390,0.282082,0.047085,0.696342,832,2141,True,0.370975,0.325367,0.696342,0.325367
8,transition_recovery,TRANS_SPRING_COMPRESSION,0.000000,0.079487,-0.330532,113,390,0.020433,0.098085,-0.292406,832,2141,True,0.038126,0.330532,0.292406,0.292406
9,rulebase_reversal,RB_PROFIT_TAKE_PROXY,0.314816,0.265772,0.269991,113,390,0.406441,0.270095,0.781418,832,2141,True,0.511427,0.269991,0.781418,0.269991


In [9]:
best_test = result_test.feature_set_scores.iloc[0].to_dict()
best_train = result_train.feature_set_scores.iloc[0].to_dict()
transfer_same_best = best_test.get('feature_set') == best_train.get('feature_set')

score_join = result_test.feature_set_scores.merge(
    result_train.feature_set_scores,
    on='feature_set',
    suffixes=('_test', '_train'),
)
score_join['both_representation_gate'] = score_join['representation_gate_test'].astype(bool) & score_join['representation_gate_train'].astype(bool)
score_join['combined_auc_min'] = score_join[['holdout_auc_min_test', 'holdout_auc_min_train']].min(axis=1)
score_join['combined_accuracy_min'] = score_join[['holdout_accuracy_min_test', 'holdout_accuracy_min_train']].min(axis=1)
score_join = score_join.sort_values(
    ['both_representation_gate', 'combined_auc_min', 'combined_accuracy_min', 'feature_set'],
    ascending=[False, False, False, True],
).reset_index(drop=True)
best_transfer = score_join.iloc[0].to_dict() if not score_join.empty else {}

if bool(best_transfer.get('both_representation_gate', False)):
    decision = 'failedfold_state_head_features_ready_for_target_recipe_no_promotion'
    next_mechanism = f"build_{best_transfer['feature_set']}_failedfold_state_head_target_recipe_then_checkpoint_screen"
    state_feature_recipe_ready = True
    target_revision_ready = False
elif bool(best_test.get('representation_gate', False)):
    decision = 'failedfold_test_state_separable_but_train_overlay_misaligned_no_promotion'
    next_mechanism = f"rebuild_failedfold_targets_around_{best_test['feature_set']}_state_head_before_training"
    state_feature_recipe_ready = False
    target_revision_ready = False
else:
    decision = 'failedfold_state_features_not_stable_expand_representation_no_promotion'
    next_mechanism = 'expand_failedfold_state_features_or_relabel_low_position_up_days_before_training'
    state_feature_recipe_ready = False
    target_revision_ready = False

fold_metrics = []
for row in source_selection['metrics']:
    if row.get('window') in ALL_FOLDS:
        fold_metrics.append(row)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'retained_soxl_harness_model_frontier',
    'direct_comparison_baseline': 'release16_remaining_failedfold_diagnostic_v1',
    'reference_baseline': source_selection.get('reference_baseline'),
    'source_failedfold_diagnostic': 'release16_remaining_failedfold_diagnostic_v1',
    'source_checkpoint_screen': 'release16_failedfold_overlay_checkpoint_screen_v1',
    'source_target_experiment': 'release16_failedfold_participation_target_overlay_v1',
    'changed_layer': ['feature', 'representation', 'target_design', 'model_selection'],
    'failed_folds': FAILED_FOLDS,
    'guard_folds': GUARD_FOLDS,
    'metrics': fold_metrics,
    'primary_blocker': 'walkforward_relative_return_gap',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'state_feature_recipe_ready': bool(state_feature_recipe_ready),
    'target_revision_ready': bool(target_revision_ready),
    'next_training_allowed_after_target_revision': False,
    'best_test_feature_set': best_test.get('feature_set'),
    'best_train_overlay_feature_set': best_train.get('feature_set'),
    'best_transfer_feature_set': best_transfer.get('feature_set'),
    'test_representation_gate': bool(best_test.get('representation_gate', False)),
    'train_overlay_representation_gate': bool(best_train.get('representation_gate', False)),
    'both_representation_gate': bool(best_transfer.get('both_representation_gate', False)),
    'transfer_same_best': bool(transfer_same_best),
    'next_mechanism': next_mechanism,
    'summary_points': [
        f"Best failed-test state group: {best_test.get('feature_set')} with AUC min {best_test.get('holdout_auc_min'):.3f} and accuracy min {best_test.get('holdout_accuracy_min'):.3f}.",
        f"Best train-overlay group: {best_train.get('feature_set')} with AUC min {best_train.get('holdout_auc_min'):.3f} and accuracy min {best_train.get('holdout_accuracy_min'):.3f}.",
        f"Best transfer group: {best_transfer.get('feature_set')} with combined AUC min {best_transfer.get('combined_auc_min'):.3f} and combined accuracy min {best_transfer.get('combined_accuracy_min'):.3f}.",
        'No actor is promoted or retained from feature mining evidence.',
    ],
    'sidecars': {key: str(path.relative_to(ROOT)) for key, path in SIDECARS.items() if key not in {'summary', 'selection', 'conclusion'}},
}
summary = dict(selection)
summary.update({
    'feature_frame_profile': PROFILE_NAME,
    'feature_packs': list(FEATURE_PACKS),
    'feature_frame_rows': int(len(feature_frame)),
    'feature_frame_columns': int(len(feature_frame.columns)),
    'candidate_feature_group_count': int(len(CANDIDATE_FEATURE_GROUPS)),
    'fold_signal_coverage': coverage.to_dict('records'),
    'test_state_summary': result_test.summary,
    'train_overlay_summary': result_train.summary,
    'feature_set_transfer_summary': score_join.to_dict('records'),
    'top_aligned_features': alignment.head(20).to_dict('records'),
    'conclusion': 'Use state-head feature mining to revise the failed-fold target recipe before another training branch; do not promote or retain an actor from this diagnostic.',
})
write_json(SIDECARS['selection'], selection)
write_json(SIDECARS['summary'], summary)
print(json.dumps(selection, indent=2)[:5000])

{
  "experiment": "release16_failedfold_state_feature_mining_v1",
  "created_at": "2026-06-14T15:50:31.280166+00:00",
  "decision": "failedfold_state_head_features_ready_for_target_recipe_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "retained_soxl_harness_model_frontier",
  "direct_comparison_baseline": "release16_remaining_failedfold_diagnostic_v1",
  "reference_baseline": "soxlbacktest_qlib_walkforward_baseline_v1:wf_pass_ytd18_15_15_02",
  "source_failedfold_diagnostic": "release16_remaining_failedfold_diagnostic_v1",
  "source_checkpoint_screen": "release16_failedfold_overlay_checkpoint_screen_v1",
  "source_target_experiment": "release16_failedfold_participation_target_overlay_v1",
  "changed_layer": [
    "feature",
    "representation",
    "target_design",
    "model_selection"
  ],
  "failed_folds": [
    "wf_2020",
    "wf_2023_2024"
  ],
  "guard_folds": [
    "wf_2022"
  ],
  "metrics": [
    {
      "avg_pos": 0.2302025320757574,
      "max_dd_pct": 16.6521

In [10]:
selection = load_json(SIDECARS['selection'])
summary = load_json(SIDECARS['summary'])
score_join = pd.read_csv(SIDECARS['test_state_feature_set_scores']).merge(
    pd.read_csv(SIDECARS['train_overlay_feature_set_scores']),
    on='feature_set',
    suffixes=('_test', '_train'),
)
coverage = pd.read_csv(SIDECARS['fold_signal_coverage'])
alignment = pd.read_csv(SIDECARS['state_transfer_feature_alignment'])

lines = [
    '# Release16 Failed-Fold State Feature Mining',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
    'Failed-fold coverage:',
]
for row in coverage.itertuples(index=False):
    lines.append(
        f"- {row.fold}: rows {int(row.rows)}, feature rows {int(row.feature_rows)}, "
        f"missed low-position up days {int(row.missed_up_days)}, label-rule match {row.label_rule_match_rate:.3f}, "
        f"return {row.return_pct:.2f}%, MaxDD {row.max_dd_pct:.2f}%."
    )

lines.extend(['', 'Feature-set gates:'])
for row in score_join.sort_values('holdout_auc_min_test', ascending=False).head(6).itertuples(index=False):
    lines.append(
        f"- {row.feature_set}: test AUC min {row.holdout_auc_min_test:.3f}, test acc min {row.holdout_accuracy_min_test:.3f}, "
        f"train-overlay AUC min {row.holdout_auc_min_train:.3f}, train-overlay acc min {row.holdout_accuracy_min_train:.3f}."
    )

lines.extend(['', 'Top state-transfer aligned features:'])
for row in alignment.head(8).itertuples(index=False):
    lines.append(
        f"- {row.feature_set}/{row.feature}: test effect {row.test_effect_size:.3f}, train effect {row.train_effect_size:.3f}, aligned={bool(row.effect_sign_aligned)}."
    )

lines.extend([
    '',
    f"Best failed-test feature set: `{selection['best_test_feature_set']}`.",
    f"Best train-overlay feature set: `{selection['best_train_overlay_feature_set']}`.",
    f"Best transfer feature set: `{selection['best_transfer_feature_set']}`.",
    f"State feature recipe ready: `{selection['state_feature_recipe_ready']}`.",
    f"Target revision ready: `{selection['target_revision_ready']}`.",
    '',
    'Risk note: this notebook uses failed test folds only for failure attribution and state-head feature mining; it does not generate eval-window training labels, train an actor, promote a model, or retain a checkpoint.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `missed_up_day_state_frame.csv`, `train_overlay_state_frame.csv`, `test_state_feature_set_scores.csv`, `test_state_fold_metrics.csv`, `test_state_feature_effects.csv`, `train_overlay_feature_set_scores.csv`, `train_overlay_fold_metrics.csv`, `train_overlay_feature_effects.csv`, `state_transfer_feature_alignment.csv`, `fold_signal_coverage.csv`, `candidate_feature_groups.json`, `selection.json`, `summary.json`.',
])
conclusion = '\n'.join(lines).rstrip() + '\n'
SIDECARS['conclusion'].write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))

# Release16 Failed-Fold State Feature Mining

Decision: `failedfold_state_head_features_ready_for_target_recipe_no_promotion`.

Failed-fold coverage:
- wf_2020: rows 252, feature rows 252, missed low-position up days 47, label-rule match 1.000, return 69.51%, MaxDD 16.65%.
- wf_2022: rows 250, feature rows 250, missed low-position up days 0, label-rule match 0.732, return -14.87%, MaxDD 15.62%.
- wf_2023_2024: rows 251, feature rows 251, missed low-position up days 66, label-rule match 1.000, return 3.05%, MaxDD 5.55%.

Feature-set gates:
- rulebase_reversal: test AUC min 0.864, test acc min 0.631, train-overlay AUC min 0.667, train-overlay acc min 0.576.
- compact_state_head: test AUC min 0.691, test acc min 0.540, train-overlay AUC min 0.662, train-overlay acc min 0.580.
- momentum_drawdown: test AUC min 0.625, test acc min 0.460, train-overlay AUC min 0.677, train-overlay acc min 0.605.
- transition_recovery: test AUC min 0.603, test acc min 0.679, train-overlay AUC min 0.609, train-overlay acc min 0.495.
- volatility_stress: test AUC min 0.542, test acc min 0.454, train-overlay AUC min 0.660, train-overlay acc min 0.437.
- soxx_relative_decay: test AUC min 0.376, test acc min 0.542, train-overlay AUC min 0.612, train-overlay acc min 0.565.

Top state-transfer aligned features:
- rulebase_reversal/RB_SELL_COMPOSITE: test effect 0.490, train effect 0.712, aligned=True.
- compact_state_head/mom_5: test effect 0.467, train effect 0.408, aligned=True.
- momentum_drawdown/mom_5: test effect 0.467, train effect 0.408, aligned=True.
- rulebase_reversal/RB_RSI6_OVERSOLD: test effect -0.400, train effect -0.360, aligned=True.
- compact_state_head/RB_TREND_REVERSAL: test effect 0.357, train effect 0.678, aligned=True.
- rulebase_reversal/RB_TREND_REVERSAL: test effect 0.357, train effect 0.678, aligned=True.
- rulebase_reversal/RB_CUMULATIVE_DROP_SEVERITY: test effect 0.329, train effect 0.713, aligned=True.
- momentum_drawdown/anchor_gap_180: test effect 0.325, train effect 0.696, aligned=True.

Best failed-test feature set: `rulebase_reversal`.
Best train-overlay feature set: `momentum_drawdown`.
Best transfer feature set: `rulebase_reversal`.
State feature recipe ready: `True`.
Target revision ready: `False`.

Risk note: this notebook uses failed test folds only for failure attribution and state-head feature mining; it does not generate eval-window training labels, train an actor, promote a model, or retain a checkpoint.

Next action: `build_rulebase_reversal_failedfold_state_head_target_recipe_then_checkpoint_screen`.

Artifacts: `missed_up_day_state_frame.csv`, `train_overlay_state_frame.csv`, `test_state_feature_set_scores.csv`, `test_state_fold_metrics.csv`, `test_state_feature_effects.csv`, `train_overlay_feature_set_scores.csv`, `train_overlay_fold_metrics.csv`, `train_overlay_feature_effects.csv`, `state_transfer_feature_alignment.csv`, `fold_signal_coverage.csv`, `candidate_feature_groups.json`, `selection.json`, `summary.json`.
